# Robust Portfolio Construction

Module: Modern Portfolio Theory

## Lesson summary

This lab extends classical mean-variance optimization with robust portfolio tools. Students compare sample covariance against shrinkage estimators, estimate CAPM beta with HAC standard errors, construct risk parity weights, and preview Hierarchical Risk Parity.

## Learning objectives

By the end of this lab, students should be able to:

- explain why sample covariance becomes unstable in small samples;
- compare Ledoit-Wolf and OAS shrinkage estimates;
- estimate CAPM beta with heteroskedasticity and autocorrelation consistent errors;
- construct a long-only risk parity portfolio;
- describe why HRP avoids full covariance matrix inversion.

## Portfolio equations

Classical minimum-variance allocation solves:

$$
\min_w w^\top\Sigma w
\quad\text{subject to}\quad
\mathbf{1}^\top w=1.
$$

CAPM beta measures exposure to the selected market proxy:

$$
\beta_i=\frac{\operatorname{Cov}(r_i-r_f,r_m-r_f)}{\operatorname{Var}(r_m-r_f)}.
$$

Risk parity targets balanced marginal contributions to portfolio volatility:

$$
\operatorname{RC}_i=\frac{w_i(\Sigma w)_i}{\sqrt{w^\top\Sigma w}}.
$$

## Setup

In [ ]:
import numpy as np
import pandas as pd

from src.portfolio_optimization import (
    annualized_mean_returns,
    capm_beta,
    global_minimum_variance_weights,
    hierarchical_risk_parity_weights,
    ledoit_wolf_covariance,
    oas_covariance,
    risk_contribution_percentages,
    risk_parity_weights,
    rolling_beta,
    sample_covariance,
)

## Simulate a small-sample asset universe

The sample has more assets than a classroom example usually needs, but fewer observations than an institutional backtest would prefer. This makes covariance estimation risk visible.

In [ ]:
rng = np.random.default_rng(52)
n_days = 180
n_assets = 8
dates = pd.bdate_range("2024-01-02", periods=n_days)

market_factor = rng.normal(0.00025, 0.010, size=n_days)
sector_factor = rng.normal(0.00010, 0.007, size=(n_days, 2))
asset_loadings = rng.uniform(0.6, 1.2, size=n_assets)
sector_map = np.array([0, 0, 0, 1, 1, 1, 0, 1])
idiosyncratic = rng.normal(0, 0.011, size=(n_days, n_assets))

return_matrix = (
    market_factor[:, None] * asset_loadings
    + sector_factor[:, sector_map] * 0.7
    + idiosyncratic
)
returns = pd.DataFrame(
    return_matrix,
    index=dates,
    columns=[f"asset_{idx + 1}" for idx in range(n_assets)],
)

returns.head()

## Sample covariance versus shrinkage

In [ ]:
expected_returns = annualized_mean_returns(returns)
sample_cov = sample_covariance(returns)
lw_cov, lw_shrinkage = ledoit_wolf_covariance(returns)
oas_cov, oas_shrinkage = oas_covariance(returns)

pd.Series(
    {
        "ledoit_wolf_shrinkage": lw_shrinkage,
        "oas_shrinkage": oas_shrinkage,
        "sample_condition_number": np.linalg.cond(sample_cov),
        "ledoit_wolf_condition_number": np.linalg.cond(lw_cov),
        "oas_condition_number": np.linalg.cond(oas_cov),
    }
)

## Minimum-variance weights under different covariance estimates

In [ ]:
gmvp_comparison = pd.concat(
    {
        "sample_covariance": global_minimum_variance_weights(sample_cov),
        "ledoit_wolf": global_minimum_variance_weights(lw_cov),
        "oas": global_minimum_variance_weights(oas_cov),
    },
    axis=1,
)

gmvp_comparison

## CAPM beta with HAC standard errors

The market proxy is the equally weighted return across the asset universe. The risk-free rate is converted to a daily rate for the regression.

In [ ]:
market_proxy = returns.mean(axis=1).rename("market_proxy")
daily_risk_free_rate = 0.065 / 252

capm_beta(
    returns["asset_1"],
    market_proxy,
    risk_free_rate=daily_risk_free_rate,
)

In [ ]:
rolling_beta(
    returns["asset_1"],
    market_proxy,
    window=60,
).dropna().tail()

## Risk parity

Risk parity avoids direct expected-return forecasts and targets equal contributions to total portfolio volatility.

In [ ]:
rp_weights = risk_parity_weights(lw_cov)
rp_risk_contribution = risk_contribution_percentages(rp_weights, lw_cov)

pd.DataFrame(
    {
        "risk_parity_weight": rp_weights,
        "risk_contribution_pct": rp_risk_contribution,
    }
)

## Hierarchical Risk Parity preview

HRP uses clustering and recursive bisection instead of a global inverse covariance matrix. It is especially useful as a robustness benchmark when classical Markowitz weights are unstable.

In [ ]:
hrp_weights = hierarchical_risk_parity_weights(returns)

pd.DataFrame(
    {
        "risk_parity": rp_weights,
        "hrp": hrp_weights,
    }
)

## Interpretation checklist

| Question | What to inspect |
| --- | --- |
| Is covariance estimation stable? | Condition number before and after shrinkage |
| Which shrinkage estimator is preferable? | Distributional assumptions and sample size |
| Is beta statistically useful? | HAC p-value and rolling beta stability |
| Does risk parity work as intended? | Risk contribution percentages |
| Does HRP differ materially? | Weight concentration and sector clustering intuition |

## Model limitations

- Shrinkage estimators stabilize covariance matrices but do not eliminate model risk in expected returns or factor structure.
- CAPM beta estimates depend on the chosen market proxy, risk-free rate, window length, and error model.
- Risk parity and HRP reduce some concentration problems, but they can still be unstable when correlations shift.